In [1]:
def clean_text(text):
    text = text.lower()  # minuscule
    
    # حذف punctuation
    for p in punctuations:
        text = text.replace(p, " ")
    
    # حذف الأرقام
    text = ''.join([char for char in text if not char.isdigit()])
    
    return text

In [3]:
def preprocess(text):
    text = clean_text(text)
    doc = nlp(text)
    
    tokens = []
    
    for token in doc:
        # نحتفظ فقط بالكلمات
        if token.is_alpha and not token.is_stop:
            tokens.append(token.lemma_)
    
    return tokens

In [11]:
import pandas as pd

fake_df = pd.read_csv('/kaggle/input/datasets/emineyetm/fake-news-detection-datasets/News _dataset/Fake.csv')
true_df = pd.read_csv('/kaggle/input/datasets/emineyetm/fake-news-detection-datasets/News _dataset/True.csv')

In [13]:
processed_docs = []
X = pd.read_csv('/kaggle/input/datasets/emineyetm/fake-news-detection-datasets/News _dataset')
for doc in X:
    tokens = preprocess(doc)
    processed_docs.append(tokens)

print(processed_docs[0])

IsADirectoryError: [Errno 21] Is a directory: '/kaggle/input/datasets/emineyetm/fake-news-detection-datasets/News _dataset'

In [5]:
vocab = {}

for doc in processed_docs:
    for word in doc:
        if word not in vocab:
            vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

Vocabulary size: 0


In [29]:
import spacy
import string
import pandas as pd
import numpy as np

# تحميل اللغة
nlp = spacy.load("en_core_web_sm")

# punctuation
punctuations = list(string.punctuation)

# تحميل البيانات
fake_df = pd.read_csv('/kaggle/input/datasets/emineyetm/fake-news-detection-datasets/News _dataset/Fake.csv')
true_df = pd.read_csv('/kaggle/input/datasets/emineyetm/fake-news-detection-datasets/News _dataset/True.csv')

# labels
fake_df['label'] = 0
true_df['label'] = 1

# دمج البيانات
data = pd.concat([fake_df, true_df], ignore_index=True)

# shuffle
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# أخذ عينة صغيرة
X = data['text'].tolist()[:2]
y = data['label'].tolist()[:5]
print(X)
print(y)

['21st Century Wire says Ben Stein, reputable professor from, Pepperdine University (also of some Hollywood fame appearing in TV shows and films such as Ferris Bueller s Day Off) made some provocative statements on Judge Jeanine Pirro s show recently. While discussing the halt that was imposed on President Trump s Executive Order on travel. Stein referred to the judgement by the 9th Circuit Court in Washington state as a  Coup d tat against the executive branch and against the constitution.  Stein went on to call the Judges in Seattle  political puppets  and the judiciary  political pawns. Watch the interview below for the complete statements and note the stark contrast to the rhetoric of the leftist media and pundits who neglect to note that no court has ever blocked any Presidential orders in immigration in the past or discuss the legal efficacy of the halt or the actual text of the Executive Order.READ MORE TRUMP NEWS AT: 21st Century Wire Trump FilesSUPPORT OUR WORK BY SUBSCRIBING 

In [16]:
def clean_text(text):
    text = text.lower()
    
    for p in punctuations:
        text = text.replace(p, " ")
    
    text = ''.join([c for c in text if not c.isdigit()])
    
    return text

In [17]:
def preprocess(text):
    text = clean_text(text)
    doc = nlp(text)
    
    tokens = []
    
    for token in doc:
        if token.is_alpha and not token.is_stop:
            tokens.append(token.lemma_)
    
    return tokens

In [18]:
processed_docs = []

for doc in X:
    processed_docs.append(preprocess(doc))

In [19]:
vocab = {}

for doc in processed_docs:
    for word in doc:
        if word not in vocab:
            vocab[word] = len(vocab)

In [20]:
def vectorize(doc, vocab):
    vector = [0] * len(vocab)
    
    for word in doc:
        if word in vocab:
            vector[vocab[word]] = 1
    
    return vector

In [1]:
Vectors = []

for doc in processed_docs:
    Vectors.append(vectorize(doc, vocab))


NameError: name 'processed_docs' is not defined

In [23]:
X_vectors = np.array(Vectors)
y_array = np.array(y)

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_vectors, y_array, test_size=0.2, random_state=42, stratify=y_array
)

In [25]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report

nb = BernoulliNB()

nb.fit(X_train, y_train)

y_pred = nb.predict(X_test)

In [26]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8
              precision    recall  f1-score   support

           0       1.00      0.60      0.75         5
           1       0.71      1.00      0.83         5

    accuracy                           0.80        10
   macro avg       0.86      0.80      0.79        10
weighted avg       0.86      0.80      0.79        10

